# Mushroom — definitieve voorbereiding zonder grafieken

Dit notebook vat de voorbereiding voor de gekozen pipeline samen. Codex maakte
deze aparte, uitvoerbare versie op verzoek van Jorre. Stap 5 (menselijke bijdragen
en opleidingspolicy) blijft door Jorre af te ronden.

Bron is de cursus-CSV met 5.000 rijen en 13 kolommen. We normaliseren namen,
verwijderen uitsluitend `jumbled_noise_0`, behouden `jumbled_noise_1` en coderen
e=0/p=1. Nulmetingen blijven nul, zoals bij de gekozen RF300. Ontbrekende waarden
blijven in de ruwe featuretabellen staan: imputation en encoding leren later
uitsluitend uit de trainingsrijen, opnieuw binnen iedere CV-fold.

We controleren bronhash, schema, duplicaten, labels en de exact gedeelde splits.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'MushroomFinal/mushroom_project_dataset.csv').exists())
sys.path.insert(0, str(ROOT))
from Final.tools.data import prepare, SOURCE_SHA256_LF, FEATURES
raw, X, y, train, test, folds = prepare()
manifest = pd.read_csv(ROOT/'MushroomFinal/comparison_correct/split_manifest.csv')
assert X.shape == (5000, 11) and not set(train) & set(test)
assert set(manifest.loc[manifest.split == 'train', 'source_row']) == set(train)
assert set(manifest.loc[manifest.split == 'test', 'source_row']) == set(test)
assert 'jumbled_noise_0' not in X and 'jumbled_noise_1' in X
print('Bronhash LF:', SOURCE_SHA256_LF)
print('Train/test:', len(train), len(test), '; CV-folds:', len(folds))
display(raw.isna().sum().rename('ontbrekend').to_frame())
display(raw['class'].value_counts().rename('aantal').to_frame())

Bronhash LF: 1f1a25f2f330458ed95ce9f6ffe3241582312a42cb21797a2b7bd5e5ad281114
Train/test: 4100 900 ; CV-folds: 5


,ontbrekend
cap_diameter,2032
stem_height,1013
stem_width,395
spore_print_color,4740
gill_color,989
habitat,1004
season,973
ring_type,577
cap_shape,406
stem_surface,3359


,aantal
class,
e,3107
p,1893


De medianen, modi en one-hot-categorieën leren alleen uit training. De testset verandert deze waarden niet. In CV wordt een nieuwe kopie van deze transformer binnen elke fold gefit.

In [2]:
from MushroomFinal.final_model_support import preprocessing
preprocessor = preprocessing(X)
encoded_train = preprocessor.fit_transform(X.iloc[train])
encoded_test = preprocessor.transform(X.iloc[test])
numeric = preprocessor.named_transformers_['num'].named_steps['impute']
assert (numeric.statistics_ == X.iloc[train][['cap_diameter','stem_height','stem_width']].median().to_numpy()).all()
assert encoded_train.shape[1] == encoded_test.shape[1]
print('Encoded train/test:', encoded_train.shape, encoded_test.shape)
print('Alle imputatie- en encodervoorzieningen zijn op training geleerd.')

Encoded train/test: (4100, 64) (900, 64)
Alle imputatie- en encodervoorzieningen zijn op training geleerd.


We bewaren ruwe gesplitste features, targets en rijnummers in een genegeerde werkmap. Zo kan ieder model zijn eigen fold-lokale preprocessing toepassen. Er worden geen grafieken of modellen gemaakt.

In [3]:
folder = ROOT/'Final/.generated/prepared'
folder.mkdir(parents=True, exist_ok=True)
for name, indices in [('train', train), ('test', test)]:
    table = X.iloc[indices].copy()
    table.insert(0, 'source_row', indices)
    table['target'] = y.iloc[indices].to_numpy()
    table.to_csv(folder/f'{name}.csv', index=False)
schema = {'source_sha256_lf': SOURCE_SHA256_LF, 'features': FEATURES,
          'train_rows':len(train), 'test_rows':len(test), 'target':{'e':0,'p':1},
          'fit_preprocessing_on':'training only, separately per CV fold'}
(folder/'schema.json').write_text(json.dumps(schema, indent=2)+'\n')
assert len(pd.read_csv(folder/'train.csv')) == 4100
assert len(pd.read_csv(folder/'test.csv')) == 900
print('Voorbereide splits gecontroleerd en opgeslagen in Final/.generated/prepared.')

Voorbereide splits gecontroleerd en opgeslagen in Final/.generated/prepared.
